In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
# 1. 데이터 불러오기, # 2. 입력/정답 분리
digits = load_digits()
X = digits.data
y = digits.target

print("X shape:", X.shape)
print("y shape:", y.shape)
# 회귀는 연속적인 숫자를 예측하므로 y값에 원-핫 인코딩을 적용하지 않음

X shape: (1797, 64)
y shape: (1797,)


In [3]:
# 3. 학습 / 테스트 데이터 분리
# y가 연속형 수치 데이터이므로 stratify=y 제거
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [4]:
# 4. 데이터 정규화
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
# 5. 모델 생성
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(64, activation="relu", input_shape=(X_train.shape[1],)))
model.add(tf.keras.layers.Dense(32, activation="relu"))
model.add(tf.keras.layers.Dense(16, activation="relu"))
model.add(tf.keras.layers.Dense(1))
# X_train.shape[1]을 사용하면 입력 특성 개수가 바껴도 수정 필요 없음
# 회귀이므로 출력층 활성화함수 제거

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [6]:
# 6. 모델 설정
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
# 회귀이므로 손실함수 mse, 매트릭스 mae 사용

In [7]:
# 7. 모델 구조 출력
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,785 (26.50 KB)

 Trainable params: 6,785 (26.50 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# 8. 모델 학습
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 11ms/step - loss: 17.7231 - mae: 3.4387 - val_loss: 8.1339 - val_mae: 2.3506
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5.8932 - mae: 2.0001 - val_loss: 4.5116 - val_mae: 1.6859
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 3.2019 - mae: 1.4050 - val_loss: 3.5461 - val_mae: 1.4227
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 2.4605 - mae: 1.1952 - val_loss: 3.1273 - val_mae: 1.2929
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 2.0157 - mae: 1.0631 - val_loss: 2.8528 - val_mae: 1.2060
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.7559 - mae: 0.9726 - val_loss: 2.5910 - val_mae: 1.1331
Epoch 7/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.5426 - mae: 0.9163 - val_loss: 2.4676 - val_mae: 1.0985
Epoch 8/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.3662 - mae: 0.8527 - val_loss: 2.2674 - val_mae: 1.0451
Epoch 9/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1.2106 - mae

In [9]:
# 9. 테스트 데이터 예측
# 회귀 모델의 predict()는 연속된 숫자를 반환하므로 .flatten()으로 1차원 변환
y_pred = model.predict(X_test).flatten()

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


In [10]:
# 10. 성능 평가
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- 회귀 성능 평가 결과 ---")
print(f"MAE (평균 절대 오차) : {mae:.4f}")
print(f"MSE (평균 제곱 오차) : {mse:.4f}")
print(f"R² Score (결정계수) : {r2:.4f}")


--- 회귀 성능 평가 결과 ---
MAE (평균 절대 오차) : 0.6297
MSE (평균 제곱 오차) : 0.8071
R² Score (결정계수) : 0.8955
